# Notebook 01: Baseline Reproduction

**Mục tiêu:** Reproduce baseline results từ paper s1 (EMNLP 2025)
- Load model Qwen2.5-7B hoặc DeepSeek-R1-Distill-7B (4-bit)
- Chạy greedy decoding (no BF) trên 50 câu MATH500
- So sánh với kết quả trong paper

**Người thực hiện:** Người A  
**Timeline:** Ngày 3–4

## 0. Setup & Imports

In [ ]:
# Colab: uncomment nếu cần install
!pip install transformers datasets accelerate bitsandbytes tqdm -q

import sys
sys.path.insert(0, '..')  # thêm experiments/ vào path

import json
import torch
from tqdm import tqdm

from models.model_loader import load_model_and_tokenizer, estimate_vram_gb
from budget_forcing import BudgetForcingDecoder, compute_all_metrics
from evaluation.run_eval import format_prompt, extract_answer, check_answer

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Check VRAM & Chọn Model

In [ ]:
# Kiểm tra VRAM estimate trước khi load
models_to_check = ['qwen2.5-3B', 'qwen2.5-7B', 'r1-distill-7B', 'r1-distill-14B']
for m in models_to_check:
    vram = estimate_vram_gb(m, load_in_4bit=True)
    print(f'{m}: ~{vram:.0f} GB (4-bit)')

# ⚠️ Chọn model phù hợp với VRAM của bạn:
# T4 16GB  → qwen2.5-7B hoặc r1-distill-7B
# A100 40GB → r1-distill-14B
# < 8GB     → qwen2.5-3B
MODEL_NAME = 'r1-distill-7B'  # ← thay đổi ở đây

## 2. Load Model

In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME, load_in_4bit=True)
decoder = BudgetForcingDecoder(model, tokenizer)
print('Model ready!')

## 3. Load MATH500 (50 samples)

In [ ]:
from datasets import load_dataset
import random

random.seed(42)
math500 = load_dataset('HuggingFaceH4/MATH-500', split='test')
indices = random.sample(range(len(math500)), 50)
samples = [math500[i] for i in indices]

print(f'Total MATH500: {len(math500)}')
print(f'Using {len(samples)} samples')
print(f'Sample 0: {samples[0]["problem"][:100]}...')

## 4. Greedy Baseline (No Budget Forcing)

In [ ]:
baseline_results = []
correct = 0

for i, sample in enumerate(tqdm(samples, desc='Baseline (no BF)')):
    question = sample['problem']
    ground_truth = sample['answer']
    
    prompt = format_prompt(question)
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    
    output = decoder.generate(
        input_ids,
        max_new_tokens=1024,
        n_wait=0,          # NO budget forcing
    )
    
    predicted = extract_answer(output['answer_text'])
    is_correct = check_answer(predicted, ground_truth)
    correct += int(is_correct)
    
    baseline_results.append({
        'idx': indices[i],
        'correct': is_correct,
        'thinking_tokens': output['thinking_tokens'],
        'predicted': predicted,
        'ground_truth': ground_truth,
    })

baseline_accuracy = correct / len(samples)
print(f'\nBaseline Accuracy: {baseline_accuracy:.1%} ({correct}/{len(samples)})')

## 5. Budget Forcing: n_wait = 1, 2, 4

In [ ]:
bf_results = {}

for n_wait in [1, 2, 4]:
    correct = 0
    run_results = []
    
    for i, sample in enumerate(tqdm(samples, desc=f'BF n_wait={n_wait}')):
        question = sample['problem']
        ground_truth = sample['answer']
        
        prompt = format_prompt(question)
        input_ids = tokenizer.encode(prompt, return_tensors='pt')
        
        output = decoder.generate(
            input_ids,
            max_new_tokens=2048,
            n_wait=n_wait,
            trigger='Wait',
        )
        
        predicted = extract_answer(output['answer_text'])
        is_correct = check_answer(predicted, ground_truth)
        correct += int(is_correct)
        run_results.append({'correct': is_correct, 'thinking_tokens': output['thinking_tokens']})
    
    acc = correct / len(samples)
    bf_results[n_wait] = {'accuracy': acc, 'details': run_results}
    print(f'n_wait={n_wait}: {acc:.1%}')

## 6. Compute Metrics & Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

compute_levels = [0, 1, 2, 4]
accuracies = [
    baseline_accuracy,
    bf_results[1]['accuracy'],
    bf_results[2]['accuracy'],
    bf_results[4]['accuracy'],
]

metrics = compute_all_metrics(compute_levels, accuracies)
print(f'Metrics: {metrics}')

# Plot accuracy vs. compute
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(compute_levels, [a * 100 for a in accuracies], 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Number of "Wait" appended (n_wait)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(f'{MODEL_NAME} on MATH500 (50 samples)\n'
             f'Control={metrics.control:.0%}, Scaling={metrics.scaling:+.1f}, '
             f'Performance={metrics.performance:.1%}', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(compute_levels)
ax.set_xticklabels(['0 (baseline)', '1×Wait', '2×Wait', '4×Wait'])

plt.tight_layout()
plt.savefig('../results/accuracy_vs_compute.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 7. Save Results

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

final = {
    'model': MODEL_NAME,
    'benchmark': 'math500',
    'n_samples': len(samples),
    'baseline_accuracy': baseline_accuracy,
    'bf_results': {str(k): v['accuracy'] for k, v in bf_results.items()},
    'metrics': {'control': metrics.control, 'scaling': metrics.scaling, 'performance': metrics.performance},
}

with open(f'../results/baseline_{MODEL_NAME.replace("/", "_")}.json', 'w') as f:
    json.dump(final, f, indent=2)

print('Results saved!')
print(json.dumps(final, indent=2))